In [ ]:
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent,
    NOTEBOOK_CWD.parent.parent,
):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.notebook_env import configure_notebook_environment

configure_notebook_environment()

import numpy as np
import pandas as pd
from pathlib import Path

from utils import (
    load_config_json,
    scale_config_to_resolution,
    load_and_preprocess_image,
    get_or_compute_vesselness,
    get_or_detect_aorta_circles,
    get_or_segment_aorta,
    detect_and_evaluate_ostia,
    visualize_aorta_with_ostia,
    segment_arteries_from_ostia,
    visualize_arteries_comparison,
)


# Análise de Casos Ruins

In [6]:
# Configurações
CONFIG_PATH = "../config/pipeline_config.json"
DATA_PATH = "/data04/home/mpmaia/ImageCAS/database/1-1000" # "/media/matheus/HD/DatasetsCCTA/ImageCAS/1-1000"
OUTPUT_PATH = "../output"
USE_CACHE = True
HTML_OUTPUT_DIR = Path(f"{OUTPUT_PATH}/cases_analysis_3d")
HTML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42

# Carregar config padrão do arquivo com uma base config mínima
base_config = {}
CONFIG = load_config_json(CONFIG_PATH, base_config)

# Carregar dados de bad cases
bad_cases_high = pd.read_csv(f"{OUTPUT_PATH}/segmentation/8.final_results/bad_cases_exports/bad_cases_test_high_res.csv")
bad_cases_mid = pd.read_csv(f"{OUTPUT_PATH}/segmentation/8.final_results/bad_cases_exports/bad_cases_test_mid_res.csv")

# Adicionar coluna de resolução
bad_cases_high['resolution'] = 'high'
bad_cases_mid['resolution'] = 'mid'

# Combinar datasets
all_bad_cases = pd.concat([bad_cases_high, bad_cases_mid], ignore_index=True)

# Configuração de amostras por tipo de erro e resolução
np.random.seed(RANDOM_SEED)
n_samples_per_error_per_resolution = 2  # 2 amostras de cada tipo de erro para cada resolução

# Agrupar por tipo de erro E resolução, e amostrar de cada combinação
selected_cases_list = []
for error_type, grp in all_bad_cases.groupby('bad_case_status'):
    high_grp = grp[grp['resolution'] == 'high']
    mid_grp = grp[grp['resolution'] == 'mid']

    high_ids = set(high_grp['image_id'].astype(int).tolist())
    mid_ids = set(mid_grp['image_id'].astype(int).tolist())
    shared_ids = sorted(list(high_ids & mid_ids))

    k_shared = min(len(shared_ids), n_samples_per_error_per_resolution)
    shared_sample_ids = []
    if k_shared > 0:
        shared_sample_ids = list(np.random.choice(shared_ids, size=k_shared, replace=False))

    for img in shared_sample_ids:
        row_high = high_grp[high_grp['image_id'].astype(int) == img].iloc[0:1]
        row_mid = mid_grp[mid_grp['image_id'].astype(int) == img].iloc[0:1]
        selected_cases_list.append(row_high)
        selected_cases_list.append(row_mid)

    for resolution, res_grp in [('high', high_grp), ('mid', mid_grp)]:
        already_ids = set(shared_sample_ids)
        remaining_needed = n_samples_per_error_per_resolution - len(shared_sample_ids)
        if remaining_needed <= 0:
            continue
        available = res_grp[~res_grp['image_id'].astype(int).isin(already_ids)]
        n_to_sample = min(remaining_needed, len(available))
        if n_to_sample > 0:
            sample = available.sample(n=n_to_sample, random_state=RANDOM_SEED)
            selected_cases_list.append(sample)

# concatenar todos os resultados selecionados
if selected_cases_list:
    selected_cases = pd.concat(selected_cases_list, ignore_index=True)
else:
    selected_cases = pd.DataFrame(columns=all_bad_cases.columns)

print(f"Total de {len(selected_cases)} casos selecionados para análise")
print(selected_cases[['image_id', 'bad_case_status', 'resolution']].sort_values(['bad_case_status', 'resolution']))

Total de 16 casos selecionados para análise
    image_id  bad_case_status resolution
0        171         low_dice       high
2        595         low_dice       high
1        171         low_dice        mid
3        595         low_dice        mid
4        916     none_correct       high
6        372     none_correct       high
5        916     none_correct        mid
7        372     none_correct        mid
8         75      one_correct       high
10       561      one_correct       high
9         75      one_correct        mid
11       561      one_correct        mid
12        86  ostia_not_found       high
14       657  ostia_not_found       high
13        86  ostia_not_found        mid
15       657  ostia_not_found        mid


In [7]:
def run_pipeline_for_case(img_id, config, data_path=None, save_base_path=None):
    """Executa o pipeline completo para uma imagem e retorna os resultados"""
    if data_path is None:
        data_path = str(DATA_PATH)

    try:
        resolution = 'high' if config['DOWNSCALE_FACTORS'] == [1, 1, 1] else 'mid'
        cache_root = save_base_path or f"{OUTPUT_PATH}/cases_analysis_cache"
        vesselness_cache_dir = f"{cache_root}/vesselness_{resolution}"
        pipeline_cache_dir = f"{cache_root}/pipeline_{resolution}"
        artery_cache_dir = f"{cache_root}/artery_{resolution}"

        run_config = dict(config)
        run_config['LOAD_CACHE'] = USE_CACHE
        run_config['SAVE_CACHE'] = USE_CACHE

        image_data = load_and_preprocess_image(img_id, data_path, run_config)
        lcc_image = image_data['lcc_image']
        label = image_data['label']
        downscale_factors = image_data['downscale_factors']
        scaled_spacing = image_data['scaled_spacing']

        del image_data

        detected_circles = get_or_detect_aorta_circles(
            img_id, lcc_image, downscale_factors, scaled_spacing,
            run_config['CIRCLE_DETECTION'], base_save_path=pipeline_cache_dir,
            load_cache=USE_CACHE, save_cache=USE_CACHE
        )

        aorta_mask = get_or_segment_aorta(
            img_id, lcc_image, detected_circles,
            run_config['LEVEL_SET'], pipeline_cache_dir,
            load_cache=USE_CACHE, save_cache=USE_CACHE
        )

        vesselness_ostios = get_or_compute_vesselness(
            img_id, lcc_image, cache_dir=vesselness_cache_dir,
            vesselness_config=run_config['VESSELNESS_AORTA'],
            load_cache=USE_CACHE, save_cache=USE_CACHE
        )

        try:
            ostia_results = detect_and_evaluate_ostia(
                aorta_mask, vesselness_ostios, label, scaled_spacing, run_config
            )
            ostia_left = ostia_results['ostia_left']
            ostia_right = ostia_results['ostia_right']
            label_artery = ostia_results['label_artery']
        except ValueError as ostia_exc:
            print(f"Erro na detecção/avaliação dos óstios para IMG {img_id}: {ostia_exc}")
            ostia_left = None
            ostia_right = None
            label_artery = None
            ostia_results = None

        del vesselness_ostios
        del detected_circles

        artery_results = segment_arteries_from_ostia(
            img_id, lcc_image, label_artery, ostia_left, ostia_right,
            run_config, artery_cache_dir
        )

        return {
            'image_id': img_id,
            'label_artery': label_artery,
            'aorta_mask': aorta_mask,
            'ostia_left': ostia_left,
            'ostia_right': ostia_right,
            'artery_results': artery_results,
            'ostia_results': ostia_results,
            'scaled_spacing': scaled_spacing
        }

    except Exception as e:
        print(f"Erro ao processar IMG {img_id}: {e}")
        import traceback
        traceback.print_exc()
        return {'success': False, 'error': str(e)}

## Visualizar Óstios + Aorta + Label

In [8]:
pipeline_results = {}

for idx, row in selected_cases.iterrows():
    img_id = int(row['image_id'])
    resolution = row['resolution']
    bad_case_status = row['bad_case_status']

    if resolution == 'high':
        CONFIG["DOWNSCALE_FACTORS"] = [1, 1, 1]
    else:
        CONFIG["DOWNSCALE_FACTORS"] = [2, 2, 1]

    scaled_config = scale_config_to_resolution(CONFIG)
    result = run_pipeline_for_case(img_id, scaled_config)
    pipeline_results[(img_id, resolution)] = result

    if not isinstance(result, dict) or result.get('success', True) is False:
        print(f"IMG {img_id} ({resolution}) - falhou: {result.get('error', 'resultado inválido') if isinstance(result, dict) else 'resultado inválido'}")
        continue

    error_type_clean = bad_case_status.lower().replace(' ', '_').replace('/', '_')
    ostia_left = result['ostia_left']
    ostia_right = result['ostia_right']
    aorta_mask = result['aorta_mask']
    label_artery = result['label_artery']
    spacing = result['scaled_spacing']

    try:
        ostia_filename = f"img_{img_id}_{resolution}_{error_type_clean}_aorta_ostios.html"
        ostia_path = HTML_OUTPUT_DIR / ostia_filename
        visualize_aorta_with_ostia(
            aorta_mask,
            ostia_left,
            ostia_right,
            spacing=spacing,
            label_mask=label_artery,
            use_physical_coords=True,
            save_html_path=ostia_path,
            display_plot=False,
        )
    except Exception as e:
        print(f"IMG {img_id} ({resolution}) - erro ao gerar HTML de óstios: {e}")

    try:
        artery_filename = f"img_{img_id}_{resolution}_{error_type_clean}_arteries_comparison.html"
        artery_path = HTML_OUTPUT_DIR / artery_filename
        artery_mask = result['artery_results']['artery_mask']
        visualize_arteries_comparison(
            label_mask=label_artery,
            predicted_mask=artery_mask,
            spacing=spacing,
            save_html_path=artery_path,
            display_plot=False,
        )
    except Exception as e:
        print(f"IMG {img_id} ({resolution}) - erro ao gerar comparação de artérias: {e}")

print(f"Visualizações concluídas para {len(selected_cases)} casos")

Parada na fatia 147: Δr=2.50 ou dist=52.88
Array salvo em: ../output/cases_analysis_cache/pipeline_high/segmented_aorta/171_mask_aorta.npy
Parada na fatia 156: Δr=7.25 ou dist=29.41
Array salvo em: ../output/cases_analysis_cache/pipeline_mid/segmented_aorta/171_mask_aorta.npy
Parada na fatia 215: Δr=6.00 ou dist=53.34
Array salvo em: ../output/cases_analysis_cache/pipeline_high/segmented_aorta/595_mask_aorta.npy
Parada na fatia 209: Δr=0.75 ou dist=31.00
Array salvo em: ../output/cases_analysis_cache/pipeline_mid/segmented_aorta/595_mask_aorta.npy
Parada na fatia 256: Δr=10.00 ou dist=52.78
Array salvo em: ../output/cases_analysis_cache/pipeline_high/segmented_aorta/916_mask_aorta.npy
Parada na fatia 182: Δr=4.08 ou dist=32.58
Array salvo em: ../output/cases_analysis_cache/pipeline_mid/segmented_aorta/916_mask_aorta.npy
Parada na fatia 187: Δr=3.00 ou dist=32.25
Array salvo em: ../output/cases_analysis_cache/pipeline_high/segmented_aorta/372_mask_aorta.npy
Parada na fatia 190: Δr=4.25 

KeyboardInterrupt: 